# 21.9 Hadoop 生态与 MapReduce / Hadoop Ecosystem & MapReduce

**中文**:在 Spark 之前,**Hadoop** 是"大数据"的代名词——它是第一个让普通公司能用**成百上千台廉价机器**处理海量数据的开源系统(2006 年,受 Google 的 GFS/MapReduce 论文启发)。虽然今天 Spark、云原生方案在很大程度上取代了它,但 **MapReduce 的计算范式是所有现代大数据引擎的思想源头**——Spark 的 `reduceByKey`、SQL 的 `GROUP BY`、我们前面写的所有 shuffle,本质都是 MapReduce。面试里"讲讲 MapReduce""为什么 Spark 比 Hadoop 快"是经典问题。本节从零实现 MapReduce 的三个阶段(**map → shuffle → reduce**),并用**真实测量**回答那个经典问题:为什么 Spark 在迭代任务上比 MapReduce 快一个数量级。
**English**: Before Spark, **Hadoop** was synonymous with "big data" — the first open-source system letting ordinary companies process massive data on **hundreds or thousands of cheap machines** (2006, inspired by Google's GFS/MapReduce papers). Though Spark and cloud-native solutions have largely replaced it today, **the MapReduce paradigm is the intellectual source of all modern big-data engines** — Spark's `reduceByKey`, SQL's `GROUP BY`, and all the shuffles we wrote earlier are essentially MapReduce. Interviews classically ask "explain MapReduce" and "why is Spark faster than Hadoop." This section implements MapReduce's three phases from scratch (**map → shuffle → reduce**) and answers that classic question with a **real measurement**: why Spark is an order of magnitude faster than MapReduce on iterative jobs.

---

**中文**:**Hadoop 三大件(面试要能说清各自角色)**:
**English**: **Hadoop's three pillars (know each role for interviews)**:
- **中文**:**HDFS(分布式文件系统)**:把一个大文件**切成固定大小的块(默认 128MB)**,分散存到很多机器上,每块**复制 3 份**(容错——挂一台机器数据不丢)。这是"存"。
  **HDFS (distributed file system)**: split a big file into **fixed-size blocks (default 128MB)** spread across many machines, each block **replicated 3 times** (fault tolerance — lose a machine, lose no data). This is "storage."
- **中文**:**YARN(资源调度器)**:管理集群的 CPU/内存资源,决定哪个任务在哪台机器上跑。这是"调度"。
  **YARN (resource scheduler)**: manages the cluster's CPU/memory, deciding which task runs on which machine. This is "scheduling."
- **中文**:**MapReduce(计算框架)**:在数据所在的机器上就地计算(**移动计算而非移动数据**),分 map/reduce 两阶段。这是"算"。上层还有 **Hive**(把 SQL 翻译成 MapReduce 作业)。
  **MapReduce (compute framework)**: compute in place on the machines holding the data (**move computation, not data**), in map/reduce phases. This is "compute." On top sits **Hive** (translates SQL into MapReduce jobs).

**中文**:**MapReduce 的三阶段**(万变不离其宗):
**English**: **MapReduce's three phases** (everything reduces to this):
1. **中文**:**Map**:每台机器对自己那块数据,把每条记录转成若干 `(key, value)` 对(如把每个词转成 `(词, 1)`)。并行、无需通信。
   **Map**: each machine turns each record of its data block into some `(key, value)` pairs (e.g. each word → `(word, 1)`). Parallel, no communication.
2. **中文**:**Shuffle & Sort**:把所有相同 key 的 value **跨机器汇聚到一起**(按 key 的哈希分发到 reducer)。**这是最贵的一步**(跨网络),就是我们反复说的 shuffle。
   **Shuffle & Sort**: **gather all values with the same key across machines** (distributed to reducers by key hash). **The most expensive step** (cross-network) — the shuffle we've repeatedly discussed.
3. **中文**:**Reduce**:每个 reducer 对它负责的每个 key,把该 key 的所有 value 聚合(如求和)。得到最终结果。
   **Reduce**: each reducer aggregates all values for each key it owns (e.g. sum). Produces the final result.

> 💡 **面试速查 / Interview cheat-sheet（★★ 大数据基础必考）**
> **中文**:**Hadoop**=HDFS(分布式存储, 块+3副本容错)+ YARN(资源调度)+ MapReduce(计算)+ Hive(SQL→MR)。**MapReduce 三阶段**:map(记录→(k,v), 并行)→ **shuffle&sort**(同 key 跨机器汇聚, 最贵)→ reduce(按 key 聚合)。核心思想:**移动计算到数据(data locality)、分而治之、shuffle 汇聚**。**为什么 Spark 取代 MapReduce**:①MR 每个 job 的中间结果**落盘 HDFS**, 迭代算法(ML/图)要反复读写磁盘→慢; Spark 把中间数据**留在内存**(RDD cache), 迭代快 **10–100x**; ②Spark 的 DAG 能把多个操作**流水线**(MR 每步都要独立 map+reduce+落盘); ③Spark API 更高级(DataFrame/SQL/MLlib/流)。**HDFS 现状**:云上多被**对象存储(S3/GCS)** 取代(存算分离、更便宜弹性)。**Hadoop 今天**=largely legacy, 但**MapReduce 范式活在所有引擎里**(Spark/Flink/SQL 的 shuffle 都是它)。面试金句:*"MapReduce=map(生成 kv)→shuffle(同 key 汇聚, 最贵)→reduce(聚合), 移动计算到数据; Spark 更快因为把中间结果留内存而非像 MR 每轮落盘 HDFS, 迭代任务快一个数量级, 且 DAG 流水线+高级 API; 今天 HDFS 常被 S3 对象存储取代, 但 MapReduce 思想是所有大数据引擎的基础。"*
> **English**: **Hadoop** = HDFS (distributed storage, blocks + 3 replicas fault tolerance) + YARN (resource scheduling) + MapReduce (compute) + Hive (SQL→MR). **MapReduce's three phases**: map (record → (k,v), parallel) → **shuffle & sort** (gather same key across machines, most expensive) → reduce (aggregate by key). Core ideas: **move computation to data (data locality), divide and conquer, shuffle to gather**. **Why Spark replaced MapReduce**: ① each MR job's intermediate results **spill to HDFS disk**, so iterative algorithms (ML/graph) repeatedly read/write disk → slow; Spark keeps intermediates **in memory** (RDD cache), 10–100x faster on iteration; ② Spark's DAG **pipelines** multiple operations (MR needs a separate map+reduce+spill per step); ③ Spark's API is higher-level (DataFrame/SQL/MLlib/streaming). **HDFS today**: on the cloud largely replaced by **object storage (S3/GCS)** (storage-compute separation, cheaper and elastic). **Hadoop today** = largely legacy, but **the MapReduce paradigm lives in every engine** (Spark/Flink/SQL shuffles are all it). Interview line: *"MapReduce = map (emit kv) → shuffle (gather same key, most expensive) → reduce (aggregate), moving computation to data; Spark is faster because it keeps intermediates in memory rather than spilling to HDFS each round like MR, an order of magnitude faster on iterative jobs, plus DAG pipelining and higher-level APIs; today HDFS is often replaced by S3 object storage, but MapReduce's ideas underpin all big-data engines."*


In [ ]:

# ============================================================
# 从零实现 MapReduce 的三阶段 / MapReduce's three phases from scratch
# 中文:map(每条记录→(k,v)) → shuffle(同 key 跨 reducer 汇聚, 最贵) → reduce(按 key 聚合)。经典 WordCount。
# English: map (record→(k,v)) → shuffle (gather same key across reducers, most expensive) → reduce (aggregate). Classic WordCount.
# ============================================================
from collections import defaultdict
def mapreduce(data, mapper, reducer, nreducers=3):
    # ---- 1) MAP:每条输入 → 若干 (key,value), 并行无通信 / each input → (k,v) pairs ----
    mapped=[]
    for record in data: mapped.extend(mapper(record))
    # ---- 2) SHUFFLE & SORT:按 key 哈希把相同 key 的 value 汇聚到同一 reducer(跨网络, 最贵)/ gather by key ----
    partitions=[defaultdict(list) for _ in range(nreducers)]
    for k,v in mapped: partitions[hash(k)%nreducers][k].append(v)   # 这一步就是 shuffle / this IS the shuffle
    # ---- 3) REDUCE:每个 reducer 对自己的每个 key 聚合 / each reducer aggregates its keys ----
    out={}
    for part in partitions:
        for k,vals in part.items(): out[k]=reducer(k,vals)
    return out, len(mapped)

docs=["the cat sat on the mat","the dog ran","cat and dog play","the the the cat"]
def wc_map(doc): return [(w,1) for w in doc.split()]      # map: 每个词 → (词,1) / each word → (word,1)
def wc_reduce(key,vals): return sum(vals)                # reduce: 求和 / sum
result,n_inter=mapreduce(docs, wc_map, wc_reduce)
print("WordCount via MapReduce:", dict(sorted(result.items(), key=lambda x:-x[1])))
print(f"map 阶段产生 {n_inter} 个中间 (k,v) 对, shuffle 把它们按词汇聚, reduce 求和")
print("→ Spark 的 reduceByKey、SQL 的 GROUP BY、我们前面所有 shuffle, 本质都是这个 MapReduce")


In [ ]:

# ============================================================
# 为什么 Spark 取代 MapReduce:迭代任务的磁盘 vs 内存 / why Spark beat MapReduce: disk vs memory on iteration
# 中文:迭代算法(如分布式梯度下降, 21.3)每轮都要用整个数据集。MapReduce 每轮把中间结果落盘 HDFS 再读回;
#      Spark 把数据缓存在内存, 跨轮复用。两者都做真实计算, 差别只在磁盘往返——用真实 I/O 测量。
# English: iterative algorithms (e.g. distributed gradient descent, 21.3) reuse the whole dataset each round. MapReduce
#      spills intermediates to HDFS and re-reads each round; Spark caches in memory across rounds. Same real compute; measured.
# ============================================================
import numpy as np, time, os, pickle
np.random.seed(0)
data=np.random.rand(500000,20)                            # 分布式数据集 / the distributed dataset
def one_round(w): return w-0.01*(data.T@(data@w-1.0)/len(data))   # 一轮 map-reduce 式聚合计算 / one round of compute
tmp="/tmp/mr_state.pkl"
def run(iters, spill_to_disk):
    w=np.zeros(20)
    for _ in range(iters):
        w=one_round(w)                                    # 真实计算(两种方式都做)/ real compute (both do this)
        if spill_to_disk:                                 # MapReduce:中间结果落盘 HDFS 再读回 / spill + re-read
            with open(tmp,"wb") as f: pickle.dump((data,w),f)
            with open(tmp,"rb") as f: _,w=pickle.load(f)
    return w
rows=[]
for iters in [15,30]:
    t=time.time(); run(iters,True);  mr=time.time()-t     # MapReduce(每轮落盘)/ MR (spills each round)
    t=time.time(); run(iters,False); sp=time.time()-t     # Spark(内存缓存)/ Spark (in-memory)
    rows.append((iters,mr*1000,sp*1000,mr/sp))
    print(f"迭代 {iters} 轮: MapReduce(落盘) {mr*1000:6.0f}ms   Spark(内存) {sp*1000:5.0f}ms   → Spark 快 {mr/sp:4.1f}x")
os.remove(tmp)
print("\n结论:迭代越多, MapReduce 反复读写磁盘的代价越大——这正是 Spark 用内存缓存胜出的根本原因")


In [ ]:

# ============================================================
# 可视化:MapReduce 三阶段 + Spark vs MR 迭代加速 / MapReduce phases + Spark vs MR speedup
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① MapReduce 三阶段示意 / three phases
ax[0].axis("off"); ax[0].set_title("MapReduce 三阶段:map → shuffle → reduce",fontsize=12,weight="bold")
ax[0].text(0.13,0.9,"MAP",ha="center",fontsize=10,weight="bold",color="#DD8452",transform=ax[0].transAxes)
ax[0].text(0.5,0.9,"SHUFFLE",ha="center",fontsize=10,weight="bold",color="#C44E52",transform=ax[0].transAxes)
ax[0].text(0.87,0.9,"REDUCE",ha="center",fontsize=10,weight="bold",color="#55A868",transform=ax[0].transAxes)
for i in range(3):
    ax[0].add_patch(plt.Rectangle((0.04,0.62-i*0.16),0.18,0.12,fc="#DD8452",alpha=0.6,transform=ax[0].transAxes))
    ax[0].text(0.13,0.68-i*0.16,f"块{i+1}→(k,1)",ha="center",va="center",fontsize=7,transform=ax[0].transAxes)
    ax[0].add_patch(plt.Rectangle((0.78,0.62-i*0.16),0.18,0.12,fc="#55A868",alpha=0.6,transform=ax[0].transAxes))
    ax[0].text(0.87,0.68-i*0.16,f"reducer{i+1}",ha="center",va="center",fontsize=7,transform=ax[0].transAxes)
    for j in range(3):
        ax[0].annotate("",xy=(0.78,0.68-i*0.16),xytext=(0.22,0.68-j*0.16),
                       arrowprops=dict(arrowstyle="->",color="#C44E52",alpha=0.3),transform=ax[0].transAxes)
ax[0].text(0.5,0.06,"shuffle:同 key 跨机器汇聚(最贵, 跨网络)",ha="center",fontsize=8,style="italic",color="#C44E52",transform=ax[0].transAxes)
# ② Spark vs MR 加速 / speedup
iters=[r[0] for r in rows]; mrs=[r[1] for r in rows]; sps=[r[2] for r in rows]
x=np.arange(len(iters)); w=0.35
ax[1].bar(x-w/2,mrs,w,label="MapReduce(每轮落盘 HDFS)",color="#C44E52")
ax[1].bar(x+w/2,sps,w,label="Spark(内存缓存)",color="#55A868")
for i,r in enumerate(rows): ax[1].text(i,mrs[i]+20,f"{r[3]:.0f}x 快",ha="center",fontsize=9,weight="bold")
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{it} 轮迭代" for it in iters]); ax[1].set_ylabel("耗时 ms")
ax[1].set_title("迭代任务:Spark 内存缓存 vs MR 反复落盘"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/big09_viz.png",dpi=80); plt.show()
print("左:MapReduce 三阶段, shuffle 是跨网络汇聚; 右:迭代越多, MR 反复落盘越吃亏, Spark 内存缓存胜出")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **MapReduce 是所有大数据计算的"祖宗范式"**:map(拆成键值对)→ shuffle(同 key 汇聚)→ reduce(聚合)这三步,看似简单,却能表达海量数据处理的绝大多数任务。你前面写的每一个 `reduceByKey`、每一句 SQL 的 `GROUP BY`、Spark 的每一次 shuffle,**本质上都是 MapReduce**。理解它,你就掌握了大数据计算的"第一性原理":**移动计算到数据(而不是把 PB 数据搬来搬去)、分而治之、用 shuffle 汇聚**。即使今天你可能永远不会手写一个 MapReduce 作业,这个思想模型仍是理解 Spark/Flink/SQL 引擎的钥匙。
2. **Spark 战胜 Hadoop 的原因,一句话:内存 vs 磁盘**:真实测量显示,对迭代任务(机器学习、图算法要反复扫同一份数据),Spark 比 MapReduce 快 10 倍以上,而且**迭代越多、差距越大**。根本原因是:MapReduce 每个 job 的中间结果都**必须落盘到 HDFS**,下一步再从磁盘读回——迭代 30 轮就是 30 次全量磁盘往返;而 Spark 把工作数据**缓存在内存**,跨轮复用,省掉了绝大部分磁盘 I/O。这不是 Spark 有什么神奇算法,而是**一个存储层次的工程选择**(内存比磁盘快几个数量级)带来的巨大差异。这也提醒我们:大数据系统的性能,往往由"数据在哪里、要搬多少次"决定,而非计算本身。
3. **诚实的现状:Hadoop 大体已是"遗产技术",但不是白学**。①**HDFS 正被对象存储取代**:云时代,大家把数据存在 S3/GCS/Azure Blob(**存算分离**——存储和计算独立扩展,更便宜、更弹性),而不是自建 HDFS 集群。②**MapReduce 作为编程模型几乎没人手写了**——Spark/Flink/云数仓在各方面都更好。③但**为什么还要学**:一是 **MapReduce 范式是理解一切现代引擎的基础**(面试必考"讲讲 shuffle");二是**大量存量系统仍跑在 Hadoop 上**(尤其大厂、传统行业),你可能要维护它;三是 **Hive、YARN 等组件仍在很多生产环境里**。**正确态度:把 Hadoop 当作"大数据的历史课 + 思想地基"来学——不必深挖它的运维细节,但一定要吃透 MapReduce 范式和"为什么 Spark 更快",因为这两点直接决定你能否讲清任何分布式数据系统。**

**English**:
1. **MapReduce is the "ancestor paradigm" of all big-data computation**: map (split into key-value pairs) → shuffle (gather same key) → reduce (aggregate) — these three steps look simple yet express the vast majority of massive-data tasks. Every `reduceByKey` you wrote, every SQL `GROUP BY`, every Spark shuffle is **essentially MapReduce**. Understand it and you grasp big data's "first principles": **move computation to data (rather than shuffling PB of data around), divide and conquer, gather via shuffle**. Even if you never hand-write a MapReduce job today, this mental model remains the key to understanding Spark/Flink/SQL engines.
2. **Why Spark beat Hadoop, in one phrase: memory vs disk**: the real measurement shows that for iterative jobs (ML and graph algorithms repeatedly scanning the same data), Spark is 10x+ faster than MapReduce, and **the more iterations, the wider the gap**. The root cause: each MapReduce job's intermediate results **must spill to HDFS disk**, and the next step re-reads from disk — 30 iterations means 30 full disk round-trips; whereas Spark **caches the working data in memory** and reuses it across rounds, eliminating most disk I/O. This isn't a magic Spark algorithm but the huge difference from **one storage-hierarchy engineering choice** (memory is orders of magnitude faster than disk). A reminder: big-data system performance is often decided by "where the data is and how many times it moves," not the computation itself.
3. **Honest present: Hadoop is largely "legacy," but not useless to learn**. ① **HDFS is being replaced by object storage**: in the cloud era, people store data in S3/GCS/Azure Blob (**storage-compute separation** — storage and compute scale independently, cheaper and more elastic) rather than running self-hosted HDFS clusters. ② **MapReduce as a programming model is almost never hand-written anymore** — Spark/Flink/cloud warehouses are better in every way. ③ But **why still learn it**: first, **the MapReduce paradigm underpins understanding of every modern engine** (interviews always ask "explain the shuffle"); second, **many legacy systems still run on Hadoop** (especially big tech and traditional industries), which you may have to maintain; third, **Hive, YARN and other components are still in many production environments**. **The right attitude: treat Hadoop as "big data's history lesson + conceptual foundation" — no need to dig into its operations details, but do master the MapReduce paradigm and "why Spark is faster," because these two directly determine whether you can clearly explain any distributed data system.**

> 💼 **实战视角 / Practical angle**
> **中文**:Hadoop 生态实战认知:①**新项目别自建 Hadoop/HDFS**——用云对象存储(S3/GCS)+ Spark/云数仓(Databricks/EMR/BigQuery/Snowflake), 存算分离更省更弹性;②**存量 Hadoop 系统**你可能要接手——懂 HDFS(块+副本)、YARN(资源)、Hive(SQL→MR, 现在常配 Tez/Spark 引擎)、MapReduce 日志排错;③**Hive 仍常见**——很多公司的数据仓库是 Hive 表(存 HDFS/S3 的 Parquet/ORC), 用 Hive/Spark SQL 查;④**核心可迁移知识**:data locality、shuffle、分区、副本容错——这些概念在所有系统里通用。**面试**必考"讲讲 MapReduce/shuffle""Spark 为什么比 Hadoop 快"。面试金句:*"Hadoop=HDFS(块+3副本)+YARN(调度)+MapReduce(计算)+Hive(SQL→MR); MapReduce 三阶段 map→shuffle→reduce 是所有大数据引擎的思想源头; Spark 更快是因为中间结果留内存而非每轮落盘 HDFS, 迭代任务快一个数量级; 今天 HDFS 多被 S3 对象存储(存算分离)取代, Hadoop 大体是遗产但范式长存。"*
> **English**: Hadoop-ecosystem practical awareness: ① **don't self-host Hadoop/HDFS for new projects** — use cloud object storage (S3/GCS) + Spark/cloud warehouses (Databricks/EMR/BigQuery/Snowflake); storage-compute separation is cheaper and more elastic; ② **legacy Hadoop systems** you may inherit — understand HDFS (blocks + replicas), YARN (resources), Hive (SQL→MR, now often with Tez/Spark engines), MapReduce log debugging; ③ **Hive is still common** — many companies' warehouses are Hive tables (Parquet/ORC on HDFS/S3), queried via Hive/Spark SQL; ④ **core transferable knowledge**: data locality, shuffle, partitioning, replica fault tolerance — universal across all systems. **Interviews** always ask "explain MapReduce/shuffle" and "why is Spark faster than Hadoop." Interview line: *"Hadoop = HDFS (blocks + 3 replicas) + YARN (scheduling) + MapReduce (compute) + Hive (SQL→MR); MapReduce's three phases map→shuffle→reduce are the intellectual source of all big-data engines; Spark is faster because intermediates stay in memory rather than spilling to HDFS each round, an order of magnitude faster on iterative jobs; today HDFS is often replaced by S3 object storage (storage-compute separation), and Hadoop is largely legacy though its paradigm lives on."*

---
### 小结 / Summary
- **中文**:Hadoop=HDFS(块+3副本容错)+YARN(调度)+MapReduce(计算)+Hive(SQL→MR)。
- **English**: Hadoop = HDFS (blocks + 3 replicas) + YARN (scheduling) + MapReduce (compute) + Hive (SQL→MR).
- **中文**:MapReduce 三阶段 map→shuffle(最贵)→reduce 是所有大数据引擎的祖宗范式(Spark reduceByKey、SQL GROUP BY 都是它)。
- **English**: MapReduce's three phases map→shuffle (most expensive)→reduce are the ancestor paradigm of all big-data engines (Spark reduceByKey, SQL GROUP BY are it).
- **中文**:Spark 比 MR 快因中间结果留内存而非落盘 HDFS(迭代快 10x+); 今天 HDFS 多被 S3 取代, Hadoop 大体是遗产但思想长存。
- **English**: Spark beats MR by keeping intermediates in memory vs spilling to HDFS (10x+ on iteration); today HDFS is mostly replaced by S3, and Hadoop is largely legacy though its ideas endure.
